In [0]:
df_bronze = spark.table("main.default.online_retail")
print("✅ Registros en Bronze:", df_bronze.count())
df_bronze.show(5)

✅ Registros en Bronze: 541909
+---------+---------+--------------------+--------+-------------------+--------------------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|           UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+--------------------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|2.549999999999999800|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|3.390000000000000000|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|2.750000000000000000|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|3.390000000000000000|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|3.390000000000000000|     17850|United Kingdom|
+---------+-------

In [0]:
df_silver = df_bronze

df_silver = df_silver \
    .filter(F.col("CustomerID").isNotNull()) \
    .filter(~F.col("InvoiceNo").startswith("C")) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0)

#  Eliminar registros sin CustomerID
#como no sabemos quien compró no podemos segmentar
# Filtrar cantidades negativas (devoluciones)
#quitamos facturas canceladas 

# Renombrar columnas al español
df_silver = df_silver \
    .withColumnRenamed("InvoiceNo",   "numero_factura") \
    .withColumnRenamed("StockCode",   "codigo_producto") \
    .withColumnRenamed("Description", "descripcion") \
    .withColumnRenamed("Quantity",    "cantidad") \
    .withColumnRenamed("InvoiceDate", "fecha_factura") \
    .withColumnRenamed("UnitPrice",   "precio_unitario") \
    .withColumnRenamed("CustomerID",  "id_cliente") \
    .withColumnRenamed("Country",     "pais")
# Calcular TotalAmount
df_silver = df_silver.withColumn(
    "TotalAmount",
    F.col("cantidad") * F.col("precio_unitario")
)

print("🥈 Silver registros:", df_silver.count())
df_silver.show(5)

# Guardar tabla
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.default.silver_sales_cleaned")

print("✅ silver_sales_cleaned guardada!")

🥈 Silver registros: 397884
+--------------+---------------+--------------------+--------+-------------------+--------------------+----------+--------------+-----------+
|numero_factura|codigo_producto|         descripcion|cantidad|      fecha_factura|     precio_unitario|id_cliente|          pais|TotalAmount|
+--------------+---------------+--------------------+--------+-------------------+--------------------+----------+--------------+-----------+
|        536365|         85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|2.549999999999999800|     17850|United Kingdom|  15.300000|
|        536365|          71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|3.390000000000000000|     17850|United Kingdom|  20.340000|
|        536365|         84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|2.750000000000000000|     17850|United Kingdom|  22.000000|
|        536365|         84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|3.390000000000000000|     17850|United 

In [0]:
bronze = spark.table("main.default.online_retail").count()
silver = spark.table("main.default.silver_transactions").count()

print(f"🥉 Bronze: {bronze:,} registros")
print(f"🥈 Silver: {silver:,} registros")
print(f"🗑️  Eliminados: {bronze - silver:,} registros sucios")


🥉 Bronze: 541,909 registros
🥈 Silver: 397,884 registros
🗑️  Eliminados: 144,025 registros sucios
